In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.nn.functional import relu, sigmoid
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

In [ ]:
np.random.seed(110007)
torch.manual_seed(110007);

In [ ]:
transform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])

train_dataset = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_dataset  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size)

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(28 * 28, 150)
        self.fc2 = nn.Linear(150, 30)
        self.fc3 = nn.Linear(30, 150)
        self.fc4 = nn.Linear(150, 28 * 28)

    def forward(self, x):
        x = self.encode(x)
        x = relu(self.fc3(x))
        x = sigmoid(self.fc4(x))
        return x

    def encode(self, x):
        x = torch.flatten(x, 1)
        x = relu(self.fc1(x))
        x = relu(self.fc2(x))
        return x

In [ ]:
def train_autoencoder(model, epochs=5):

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model size: {total_params}')
    
    train_losses = []
    
    for e in range(epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            X, _ = data
            optimizer.zero_grad()
            X_hat = model(X)
            loss = criterion(X_hat, torch.flatten(X, 1))
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        epoch_loss = running_loss / len(train_dataset)
        train_losses.append(epoch_loss)
        
        print(f'[Epoch {e+1}] Loss: {epoch_loss:.4g}')

def test_autoencoder(model):
    loss = 0.0 
    criterion = nn.MSELoss()
    with torch.no_grad():
        for data in test_loader:
            X, _ = data
            X_hat = model(X)
            loss += criterion(X_hat, torch.flatten(X, 1)).item()
    loss /= len(test_dataset)
    print(f'Loss of the network on the test images: {loss:.4g}')

In [ ]:
autoencoder = Autoencoder()
train_autoencoder(autoencoder, epochs=4)

In [ ]:
test_autoencoder(autoencoder)

In [ ]:
from torch.utils.data import Subset

y_test = test_dataset.targets
idx = torch.tensor([torch.where(y_test == i)[0][0].item() for i in range(10)])

sub = Subset(test_dataset, idx)
reconstructed = []

with torch.no_grad():
    for X, _ in sub:
        reconstructed.append(autoencoder(X))

plt.figure(figsize=(15, 3))

for i, (X, _)  in enumerate(sub):
    X, _ = test_dataset[idx[i]]
    X_hat = reconstructed[i]
    ax1 = plt.subplot(2, 10, i + 1)
    plt.imshow(X.reshape(28, 28), vmin=0, vmax=1, cmap='grey')
    plt.tight_layout()
    ax2 = plt.subplot(2, 10, i + 11)
    plt.imshow(X_hat.reshape(28, 28), vmin=0, vmax=1, cmap='grey')
        
    if i == 0:
        ax1.set_ylabel('Original')
        ax1.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
        ax2.set_ylabel(f'Reconstrucción')
        ax2.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
    else:
        ax1.axis('off')
        ax2.axis('off')

plt.tight_layout()
plt.savefig("../informe/img/ej7/reconstructions.svg")
plt.show()

In [ ]:
with torch.no_grad():
    enc_train_dataset = [(autoencoder.encode(x), y) for x,y in train_dataset]
enc_train_loader = DataLoader(enc_train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
class AEClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(30, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = relu(self.fc(x))
        x = self.fc2(x)
        return x

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(28 * 28, 10)

    def forward(self, x):
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

In [ ]:
def train(model, train_loader, epochs=5):

    optimizer = optim.Adam(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model size: {total_params}')
    
    train_losses = []
    
    for e in range(epochs):
        running_loss = 0.0
        for i, data in enumerate(train_loader, 0):
            X, y = data
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
    
            running_loss += loss.item()
    
        epoch_loss = running_loss / len(train_dataset)
        train_losses.append(epoch_loss)
        
        print(f'[Epoch {e+1}] Loss: {epoch_loss:.4f}')

def test(model, test_loader, encoder=None):
    correct = 0
    total = 0

    with torch.no_grad():
        for data in test_loader:
            X, y = data
            if encoder is not None:
                X = encoder.encode(X)
            output = model(X)
            _, y_hat = torch.max(output.data, 1)
            total += y.size(0)
            correct += (y_hat == y).sum().item()
    
    print(f'\nAccuracy of the network on the {total} test images: {100 * correct / total} %')

In [ ]:
ae_classifier = AEClassifier()

start = time.time()
train(ae_classifier, enc_train_loader, epochs=10)
end = time.time()
print(f'Training took: {end - start:.2f} sec')

In [ ]:
test(ae_classifier, test_loader, encoder=autoencoder)

In [ ]:
mlp_classifier = MLPClassifier()

start = time.time()
train(mlp_classifier, train_loader, epochs=10)
end = time.time()
print(f'Training took: {end - start:.2f} sec')

test(mlp_classifier, test_loader)